# Stage 1 — MIMIC-IV cohort and Sepsis-3 labels

Run this notebook only in your credentialed Kaggle/Colab environment. Patient-level MIMIC-IV data and derivatives must stay in that environment and under the gitignored `data/` directories. Never display cohort rows in a saved notebook, upload them to an API, or commit them.

Locked definition: first ICU stay, adults, observation window `[0, 6]` h, positive onset `(6, 30]` h, and negative only with at least 30 h follow-up and no onset in `[0, 30]` h.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from google.cloud import bigquery
from src.cohort import (
    COHORT_SQL_FILE,
    CONSORT_SQL_FILE,
    estimate_query_bytes,
    run_stage1,
)
from src.config import DATA_INTERIM, N_HOURS, M_HOURS

assert (N_HOURS, M_HOURS) == (6, 24)

## Connect with your own billing project

Set `GOOGLE_CLOUD_PROJECT` in the notebook environment to the Google Cloud project linked to your credentialed PhysioNet account. Authentication is intentionally not stored in this repository. In Colab, authenticate with the standard Google authentication UI before running this cell.

In [ ]:
billing_project = os.environ['GOOGLE_CLOUD_PROJECT']
client = bigquery.Client(project=billing_project)
print('BigQuery client ready; no patient data queried yet.')

## Dry run

This estimates scan size without returning data. Check the estimate before executing the protected queries.

In [ ]:
cohort_bytes = estimate_query_bytes(client, COHORT_SQL_FILE)
consort_bytes = estimate_query_bytes(client, CONSORT_SQL_FILE)
print(f'Cohort query estimate: {cohort_bytes / 2**30:.3f} GiB')
print(f'CONSORT query estimate: {consort_bytes / 2**30:.3f} GiB')

## Execute Stage 1

The patient-level cohort is written directly to `data/processed/cohort_mimiciv.parquet`. This cell only displays aggregate counts.

In [ ]:
artifacts = run_stage1(client)
display(artifacts.counts)
print(f'Protected cohort saved to: {artifacts.cohort_path}')
print(f'Aggregate counts saved to: {artifacts.counts_path}')
print(f'Flow figure saved to: {artifacts.figure_path}')

## Protected manual audit sample

This creates the required random 5-positive/5-negative audit file without displaying identifiers or rows. Open it only inside the controlled environment, verify onset offsets against labels, then keep it under `data/interim/` (gitignored).

In [ ]:
import pandas as pd

cohort = pd.read_parquet(artifacts.cohort_path)
audit_sample = pd.concat(
    [
        cohort.loc[cohort['label'].eq(1)].sample(n=5, random_state=42),
        cohort.loc[cohort['label'].eq(0)].sample(n=5, random_state=42),
    ],
    ignore_index=True,
)
audit_path = DATA_INTERIM / 'cohort_label_audit_sample.parquet'
audit_path.parent.mkdir(parents=True, exist_ok=True)
audit_sample.to_parquet(audit_path, index=False)
del audit_sample
print(f'Protected 10-row audit sample saved to: {audit_path}')

## Aggregate acceptance checks

Do not proceed to feature engineering until the manual audit is complete and these aggregate checks are sensible.

In [ ]:
counts = artifacts.counts.set_index('stage_code')['stay_count']
assert counts['final_cohort'] == counts['positive'] + counts['negative']
assert counts['positive'] > 0 and counts['negative'] > 0
prevalence = counts['positive'] / counts['final_cohort']
print(f'Positive prevalence: {prevalence:.1%}')
if not 0.35 <= prevalence <= 0.48:
    print('Review cohort logic and source-version differences: prevalence is outside the expected sanity-check range.')
else:
    print('Aggregate Stage-1 checks passed. Complete the protected manual audit before acceptance.')
del cohort